# 第10章: 事前学習済み言語モデル（GPT型）

本章では、GPT型（Transformerのデコーダ型）の事前学習済みモデルを利用して、言語生成、評判分析器（ポジネガ分類器）の構築、ファインチューニング、強化学習などに取り組む。

## 90. 次単語予測

“The movie was full of"に続くトークン（トークン列ではなく一つのトークンであることに注意せよ）として適切なもの上位10個と、その確率（尤度）を求めよ。ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

prompt = "The movie was full of"

inputs = tokenizer(prompt, return_tensors="pt")

print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print(inputs["input_ids"][0].tolist())

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0, -1]
probs = torch.softmax(logits, dim=-1)

top_probs, top_ids = torch.topk(probs, 10)

for token_id, prob in zip(top_ids, top_probs):
    token = tokenizer.convert_ids_to_tokens([token_id.item()])[0]
    text = tokenizer.decode([token_id.item()])
    print(f"{token_id.item():5d}\t{repr(token):12s}\t{repr(text):10s}\t{prob.item():.6f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

['The', 'Ġmovie', 'Ġwas', 'Ġfull', 'Ġof']
[464, 3807, 373, 1336, 286]
14532	'Ġjokes'    	' jokes'  	0.021892
 1049	'Ġgreat'    	' great'  	0.018644
22051	'Ġlaughs'   	' laughs' 	0.011524
 2089	'Ġbad'      	' bad'    	0.010874
24072	'Ġsurprises'	' surprises'	0.010667
10288	'Ġreferences'	' references'	0.010528
 1257	'Ġfun'      	' fun'    	0.009992
14733	'Ġhumor'    	' humor'  	0.007415
  366	'Ġ"'        	' "'      	0.007408
  262	'Ġthe'      	' the'    	0.006709


## 91. 続きのテキストの予測

“The movie was full of"に続くテキストを複数予測せよ。このとき、デコーディングの方法や温度パラメータ（temperature）を変えながら、予測される複数のテキストの変化を観察せよ。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

prompt = "The movie was full of"
inputs = tokenizer(prompt, return_tensors="pt")

def generate_text(
    do_sample=False,
    temperature=1.0,
    top_k=None,
    top_p=None,
    num_return_sequences=5,
):
    kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "max_new_tokens": 30,
        "num_return_sequences": num_return_sequences,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        kwargs["temperature"] = temperature
        if top_k is not None:
            kwargs["top_k"] = top_k
        if top_p is not None:
            kwargs["top_p"] = top_p

    outputs = model.generate(**kwargs)

    for i, output in enumerate(outputs, 1):
        text = tokenizer.decode(output, skip_special_tokens=True)
        print(f"[{i}] {text}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
generate_text(do_sample=False, num_return_sequences=1)

[1] The movie was full of jokes and jokes about how the movie was a joke. It was a joke about how the movie was a joke. It was a joke about how the


In [ ]:
generate_text(
    do_sample=True,
    temperature=0.7,
    num_return_sequences=5,
)

[1] The movie was full of humor and action. I was surprised when the movie got a decent trailer, but the only reason I was surprised is because the movie is so good.
[2] The movie was full of bad puns, bad language, and an inability to think through the consequences of its actions. It was in some ways a "bad" movie,
[3] The movie was full of strange and disturbing images, but it never really touched on the real story behind the movie. It was a really good movie. I really loved it."
[4] The movie was full of great comedy and humor but was also full of some of the worst violence ever filmed. This film, while not as bad as some of its sequels,
[5] The movie was full of laughs. I remember the movie as a little bit of a comedy, but not so much that it was a big movie. It was an action movie


In [ ]:
generate_text(
    do_sample=True,
    temperature=1.0,
    num_return_sequences=5,
)

[1] The movie was full of humor and humor. I watched it once, I was really bored. I'm going to watch this again. I'm going to watch this."

[2] The movie was full of good friends or family members who have never before seen a horror movie. It was a huge celebration and a fun thing to have. I always remember them
[3] The movie was full of controversy and a big win for the company, as Fox and Warner Bros. both saw a huge uptick in box offices compared to 2012.

A
[4] The movie was full of twists, and I think the most memorable was when the kids start getting angry at you so hard they get fired from high school. In an age when
[5] The movie was full of laughs, a good mix of horror and a real sense of humor.

Here is the full review:

The best way to appreciate my


In [ ]:
generate_text(
    do_sample=True,
    temperature=1.0,
    top_k=50,
    num_return_sequences=5,
)

[1] The movie was full of character-driven plotlines, that were just a couple of lines of dialogue. It wasn't really just a matter of saying a message was "Hey
[2] The movie was full of jokes but was a little goofy. The movie went down well for awhile and was not to be missed.

Now you can just watch it and
[3] The movie was full of allusion to a very powerful magic trick: that of teleportation—the trick of being able to pull a person or object on their own into a position
[4] The movie was full of jokes and jokes and jokes, so I'm pleased and glad it was the way it was," he said. When the movie was about the death of
[5] The movie was full of references to the Bible.

It was not until 2005, with the final film of the project, that it made its way to theaters, where


## 92. 予測されたテキストの確率を計算

“The movie was full of"に続くテキストを予測し、生成された各単語の尤度を表示せよ（生成されるテキストが長いと出力が読みにくくなるので、適当な長さで生成を打ち切るとよい）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "sshleifer/tiny-gpt2"  # 軽量版

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

prompt = "The movie was full of"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs.sequences[0]
prompt_len = inputs["input_ids"].shape[1]
new_token_ids = generated_ids[prompt_len:]

print("生成文:")
print(tokenizer.decode(generated_ids, skip_special_tokens=True))
print()

print("位置\tID\tトークン\t文字列\t尤度")
for i, token_id in enumerate(new_token_ids):
    scores = outputs.scores[i][0]
    probs = torch.softmax(scores, dim=-1)

    prob = probs[token_id].item()
    token = tokenizer.convert_ids_to_tokens([token_id.item()])[0]
    text = tokenizer.decode([token_id.item()])

    print(f"{i+1}\t{token_id.item()}\t{repr(token)}\t{repr(text)}\t{prob:.6f}")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


生成文:
The movie was full of factors factors factors factors factors factors factors factors factors factors

位置	ID	トークン	文字列	尤度
1	5087	'Ġfactors'	' factors'	0.000020
2	5087	'Ġfactors'	' factors'	0.000022
3	5087	'Ġfactors'	' factors'	0.000022
4	5087	'Ġfactors'	' factors'	0.000022
5	5087	'Ġfactors'	' factors'	0.000022
6	5087	'Ġfactors'	' factors'	0.000022
7	5087	'Ġfactors'	' factors'	0.000022
8	5087	'Ġfactors'	' factors'	0.000022
9	5087	'Ġfactors'	' factors'	0.000022
10	5087	'Ġfactors'	' factors'	0.000022


## 93. パープレキシティ

適当な文を準備して、事前学習済み言語モデルでパープレキシティを測定せよ。例えば、

+ The movie was full of surprises
+ The movies were full of surprises
+ The movie were full of surprises
+ The movies was full of surprises

の4文に対して、パープレキシティを測定して観察せよ（最後の2つの文は故意に文法的な間違いを入れた）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "sshleifer/tiny-gpt2"  # 重い場合は軽量版
# model_name = "gpt2"              # 可能ならこちら

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

sentences = [
    "The movie was full of surprises",
    "The movies were full of surprises",
    "The movie were full of surprises",
    "The movies was full of surprises",
]

def calc_perplexity(sentence):
    enc = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            labels=enc["input_ids"],
        )

    loss = outputs.loss
    ppl = torch.exp(loss)

    return loss.item(), ppl.item()

for s in sentences:
    loss, ppl = calc_perplexity(s)
    print(f"{s}")
    print(f"  loss = {loss:.4f}")
    print(f"  perplexity = {ppl:.4f}")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


The movie was full of surprises
  loss = 10.8286
  perplexity = 50440.5312
The movies were full of surprises
  loss = 10.8212
  perplexity = 50068.9102
The movie were full of surprises
  loss = 10.8287
  perplexity = 50449.9609
The movies was full of surprises
  loss = 10.8210
  perplexity = 50059.6016


## 94. チャットテンプレート

"What do you call a sweet eaten after dinner?"という問いかけに対する応答を生成するため、チャットテンプレートを適用し、言語モデルに与えるべきプロンプトを作成せよ。また、そのプロンプトに対する応答を生成し、表示せよ。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

messages = [
    {
        "role": "user",
        "content": "What do you call a sweet eaten after dinner?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("=== Prompt ===")
print(prompt)

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("=== Response ===")
print(response)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

=== Prompt ===
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What do you call a sweet eaten after dinner?<|im_end|>
<|im_start|>assistant

=== Response ===
A sweet eaten after dinner is called a "dessert." It's a delightful combination of sweet and savory elements, often featuring a variety of fruits,


## 95. マルチターンのチャット

問題94で生成された応答に対して、追加で"Please give me the plural form of the word with its spelling in reverse order."と問いかけたときの応答を生成・表示せよ。また、その時に言語モデルに与えるプロンプトを確認せよ。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

messages = [
    {
        "role": "user",
        "content": "What do you call a sweet eaten after dinner?"
    },
    {
        "role": "assistant",
        "content": "A sweet eaten after dinner is called a dessert. It's a delightful combination of sweet and savory elements, often featuring a variety of fruits,"
    },
    {
        "role": "user",
        "content": "Please give me the plural form of the word with its spelling in reverse order."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("=== Prompt ===")
print(prompt)

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("=== Response ===")
print(response)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

=== Prompt ===
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What do you call a sweet eaten after dinner?<|im_end|>
<|im_start|>assistant
A sweet eaten after dinner is called a dessert. It's a delightful combination of sweet and savory elements, often featuring a variety of fruits,<|im_end|>
<|im_start|>user
Please give me the plural form of the word with its spelling in reverse order.<|im_end|>
<|im_start|>assistant

=== Response ===
A dessert is called a dessert.


## 96. プロンプトによる感情分析

事前学習済み言語モデルで感情分析を行いたい。テキストを含むプロンプトを事前学習済み言語モデルに与え、（ファインチューニングは行わずに）テキストのポジネガを予測するという戦略で、[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)の開発データにおける正解率を測定せよ。

In [1]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-06-11 01:11:58--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.172.170.126, 18.172.170.31, 18.172.170.125, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.172.170.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip.1’

SST-2.zip.1         100%[===================>]   7.09M  --.-KB/s    in 0.1s    

2026-06-11 01:11:58 (62.7 MB/s) - ‘SST-2.zip.1’ saved [7439277/7439277]

Archive:  SST-2.zip
replace SST-2/dev.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [3]:
import pandas as pd

train_df = pd.read_csv("SST-2/train.tsv", sep="\t")
dev_df = pd.read_csv("SST-2/dev.tsv", sep="\t")

from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
from tqdm import tqdm

# モデルとトークナイザーの準備
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# GPUが使えるなら使う
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ゼロショットで感情を分類する関数
def classify_sentiment(text):
    prompt = f"Review: {text}\nSentiment:"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # 選択肢トークン列
    pos_ids = tokenizer.encode(" Positive", return_tensors="pt")[0][1:].to(device)
    neg_ids = tokenizer.encode(" Negative", return_tensors="pt")[0][1:].to(device)

    # Positive の尤度
    pos_logprob = 0.0
    current_input = input_ids.clone()
    for token_id in pos_ids:
        with torch.no_grad():
            output = model(current_input)
        log_probs = torch.nn.functional.log_softmax(output.logits[:, -1, :], dim=-1)
        pos_logprob += log_probs[0, token_id].item()
        current_input = torch.cat([current_input, token_id.view(1, 1)], dim=1)

    # Negative の尤度
    neg_logprob = 0.0
    current_input = input_ids.clone()
    for token_id in neg_ids:
        with torch.no_grad():
            output = model(current_input)
        log_probs = torch.nn.functional.log_softmax(output.logits[:, -1, :], dim=-1)
        neg_logprob += log_probs[0, token_id].item()
        current_input = torch.cat([current_input, token_id.view(1, 1)], dim=1)

    return 1 if pos_logprob > neg_logprob else 0

# devデータで精度を評価
correct = 0
total = 0

for i, row in tqdm(dev_df.iterrows(), total=len(dev_df)):
    sentence = row["sentence"]
    label = row["label"]
    pred = classify_sentiment(sentence)
    if pred == label:
        correct += 1
    total += 1

accuracy = correct / total
print(f"\nZero-shot accuracy on SST-2 dev set: {accuracy:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

100%|██████████| 872/872 [00:00<00:00, 1438.20it/s]


Zero-shot accuracy on SST-2 dev set: 0.4908


## 97. 埋め込みに基づく感情分析

事前学習済み言語モデルでテキストをベクトルで表現（エンコード）し、そのベクトルにフィードフォワード層を通すことで極性ラベルを予測するモデルを学習せよ。

In [6]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from torch.optim import AdamW
from tqdm import tqdm

# 1. データ読み込み
train_df = pd.read_csv("SST-2/train.tsv", sep="\t")

# 2. トークナイザー準備（パディングトークンを追加）
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2にはpad_tokenがないため代用

# 3. Dataset定義
class SST2Dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.data.iloc[idx]['sentence']
        label = self.data.iloc[idx]['label']
        encoded = self.tokenizer(
            sentence,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(label)
        }

train_dataset = SST2Dataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 4. GPT2 + 分類器
class GPT2ForClassification(nn.Module):
    def __init__(self):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.gpt2.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)
        # 最後のトークンの出力（GPTは右から左への予測モデルなのでこれで良い）
        last_token_output = outputs.last_hidden_state[:, -1, :]
        logits = self.classifier(self.dropout(last_token_output))
        return logits

# 5. 学習設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT2ForClassification().to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

# 6. 学習ループ（1エポック）
model.train()
for batch in tqdm(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()
    logits = model(input_ids, attention_mask)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()

print("✅ GPTベース分類モデルの学習が完了しました。")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  3%|▎         | 114/4210 [00:25<14:59,  4.55it/s]


KeyboardInterrupt: 

## 98. ファインチューニング

問題96のプロンプトに対して、正解の感情ラベルをテキストの応答として返すように事前学習済みモデルをファインチューニングせよ。

In [4]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.optim import AdamW
from tqdm import tqdm

# デバイス設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# データ読み込み
train_df = pd.read_csv("SST-2/train.tsv", sep="\t")
dev_df = pd.read_csv("SST-2/dev.tsv", sep="\t")

# ラベルマッピング
label_map = {0: "negative", 1: "positive"}
train_df["label_text"] = train_df["label"].map(label_map)

# トークナイザー・モデル準備
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # pad_token未定義のため設定
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))
model.to(device)
model.train()

# データセット定義
class SST2Dataset(Dataset):
    def __init__(self, df, tokenizer, max_length=64):
        self.texts = [
            f"User: {row['sentence']}\nAssistant: {row['label_text']}"
            for _, row in df.iterrows()
        ]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()

        # GPT2の言語モデルトレーニングでは、labelsはinput_idsそのまま
        labels = input_ids.clone()
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

# データローダー
train_dataset = SST2Dataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# 最適化手法
optimizer = AdamW(model.parameters(), lr=5e-5)

# 学習ループ
epochs = 2
for epoch in range(epochs):
    total_loss = 0
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch + 1}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch 1: 100%|██████████| 8419/8419 [26:24<00:00,  5.31it/s, loss=0.631]


Epoch 1 Average Loss: 0.7025


Epoch 2: 100%|██████████| 8419/8419 [26:19<00:00,  5.33it/s, loss=0.413]

Epoch 2 Average Loss: 0.4666


## 99. 選好チューニング

問題96のプロンプトに対して、正解の感情ラベルを含むテキストを望ましい応答、間違った感情ラベルを含むテキストを望ましくない応答として、事前学習済み言語モデルを選好チューニング (preference tuning) を実施せよ。選好チューニングのアルゴリズムとしては、近傍方策最適化 (PPO: Proximal Policy Optimization) や直接選好最適化 (DPO: Direct Preference Optimization) などが考えられる。


In [2]:
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from copy import deepcopy

# Preference Dataset作成
pref_data = []

for _, row in train_df.iterrows():
    prompt = f"User: {row['sentence']}\nAssistant:"

    if row["label"] == 1:
        chosen = " positive"
        rejected = " negative"
    else:
        chosen = " negative"
        rejected = " positive"

    pref_data.append({
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected
    })

pref_dataset = Dataset.from_list(pref_data)

ModuleNotFoundError: No module named 'trl'

In [1]:
# SFT後モデルをコピーして参照モデルにする
ref_model = deepcopy(model)

dpo_config = DPOConfig(
    output_dir="./dpo_result",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=1e-6,
    beta=0.1,
    report_to="none"
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    processing_class=tokenizer,
    train_dataset=pref_dataset
)

trainer.train()

NameError: name 'deepcopy' is not defined